In [0]:
from pyspark.sql.functions import input_file_name, regexp_extract, col, sha2, concat_ws
from pyspark.sql.types import StructType, StructField, StringType

reviews_path = "/Volumes/workspace/it3388/raw_data/game_reviews/part_*/*.csv"

review_schema = StructType([
    StructField("user", StringType(), True),
    StructField("playtime", StringType(), True),
    StructField("post_date", StringType(), True),
    StructField("helpfulness", StringType(), True),
    StructField("review", StringType(), True),
    StructField("recommend", StringType(), True),
    StructField("early_access_review", StringType(), True),
])

reviews_df = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .schema(review_schema)
    .csv(reviews_path)
    .select(
        "*",
        col("_metadata.file_path").alias("source_file"),
        col("_metadata.file_name").alias("source_file_name")
    )
    .withColumn("app_id", regexp_extract(col("source_file_name"), r"^(\d+)_(\d+)\.csv$", 1))
    .withColumn("source_total_reviews", regexp_extract(col("source_file_name"), r"^(\d+)_(\d+)\.csv$", 2))
)

reviews_df = reviews_df.withColumn(
    "review_id",
    sha2(concat_ws("||", col("app_id"), col("user"), col("post_date"), col("review")), 256)
)

display(reviews_df)

In [0]:
reviews_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.it3388.bronze_steam_reviews")

In [0]:
reviews_df = spark.table("workspace.it3388.bronze_steam_reviews")
print(reviews_df.count())